# CP2/CP3 – Fine-tuning LoRA em GPU NVIDIA (Colab)

Este notebook roda o projeto inteiro em uma **GPU NVIDIA com CUDA** do Google Colab, para quem não tem GPU NVIDIA local.

**Como usar no VSCode:** instale a extensão **Google Colab** (publisher: Google) e a extensão Jupyter, abra este arquivo, clique em **Select Kernel → Colab** e escolha um runtime com GPU (faça login com a conta Google). No navegador (colab.research.google.com), use *Ambiente de execução → Alterar tipo → GPU*.

Execute as células **em ordem**. A GPU gratuita não é garantida: confira o nome da placa na primeira célula.

## 1. Conferir a GPU (print de evidência)

In [ ]:
!nvidia-smi
import torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA", torch.version.cuda, "| PyTorch", torch.__version__)
else:
    raise SystemExit("Sem GPU: troque o runtime/kernel para um com GPU e rode de novo.")

## 2. Baixar o código do GitHub

Troque `REPO_URL` pelo link do repositório do grupo (suba o código para o GitHub antes). Se o repositório for privado, use um token: `https://<TOKEN>@github.com/usuario/repo.git` (não commite o token).

In [ ]:
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"   # <-- TROCAR
import os
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!ls

## 3. Instalar dependências
O PyTorch com CUDA já vem instalado no Colab; aqui entram só as demais bibliotecas.

In [ ]:
!pip install -q -r requirements.txt

## 4. Teste de GPU do projeto

In [ ]:
!python src/check_gpu.py

## 5. Preparar os dados

In [ ]:
!python src/prepare_data.py

## 6. Modelo base vs. um treino de teste

Se der `out of memory`, troque o modelo por `Qwen/Qwen2.5-0.5B-Instruct` em **todas** as células (`--model`).

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ou "Qwen/Qwen2.5-0.5B-Instruct" se faltar memória
!python src/evaluate.py --tag base --model {MODEL}
!python src/train.py --name baseline --model {MODEL}
!python src/evaluate.py --adapter outputs/baseline/adapter --tag baseline --model {MODEL}

## 7. Bateria de experimentos (diferentes parâmetros)

Use `--quick` para a versão rápida (3 configurações). Sem `--quick` roda as 7 configurações e o teste de temperatura. Se a sessão cair, rode de novo: o que já foi salvo em `results/` é regenerado.

In [ ]:
!python src/run_experiments.py --quick --model {MODEL}

## 8. Gráficos e tabelas de resultados

In [ ]:
!python src/plot_results.py
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv("results/experiments.csv"))
display(pd.read_csv("results/gen_sweep.csv"))
display(Image("docs/img/experimentos_f1.png"))
display(Image("docs/img/curvas_loss.png"))

## 9. Respostas para a análise qualitativa

Compare o modelo base com o fine-tuned (cole 2–3 exemplos no README).

In [ ]:
base = pd.read_csv("results/eval_base.csv")
best = pd.read_csv("results/eval_best_T0.0.csv")
for i in range(3):
    print("="*80); print("RELATO:", base.instruction[i])
    print("\n--- BASE ---\n", base.prediction[i])
    print("\n--- FINE-TUNED ---\n", best.prediction[i])
    print("\n--- REFERÊNCIA ---\n", base.reference[i])

## 10. Frontend (Gradio)

Roda o app com link público (`--share`). Abra o link `https://....gradio.live` que aparecer, teste um relato e tire o print. Para encerrar, interrompa a célula.

In [ ]:
!python src/app.py --share --model {MODEL}

## 11. Salvar os resultados

Os arquivos de `results/` e `docs/img/` precisam ir para o repositório. Duas opções: (a) baixar o zip abaixo; (b) copiar para o Google Drive. Depois coloque as pastas no repositório e dê commit.

In [ ]:
!zip -qr resultados.zip results docs/img
try:
    from google.colab import files
    files.download("resultados.zip")
except Exception as e:
    print("Download automático indisponível aqui:", e)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        !mkdir -p /content/drive/MyDrive/cp2_resultados && cp -r results docs/img /content/drive/MyDrive/cp2_resultados/
        print("Copiado para o Google Drive: cp2_resultados")
    except Exception as e2:
        print("Drive indisponível:", e2, "- copie o conteúdo exibido nas células acima.")